# Demo multiagente -- Agente de produccion de reels 3D

Avance inicial para la materia de Agentes Inteligentes. Este notebook corre
un sistema de **6 agentes especializados** (Orquestador, Investigacion,
Guion, Voz, Visual/3D y QA) que colaboran, bajo un estado compartido
(`ReelState`), y termina produciendo un **MP4 real con voz real**.

**Alcance de esta demo** (ver README.md para el detalle):
- Investigacion y Visual usan herramientas mockeadas (simulan su resultado).
- **Voz es real**: el agente llama a OmniVoice de verdad (subproceso, venv
  propio) -- requiere cerrar WSL y Chrome antes de correr esta celda, ver
  la advertencia en la seccion 5.
- **El render final es real**: `remotion render` sobre una composicion
  generica (`AgenteReel`, en el repo publico `3d-animations-showcase`) que
  lee el guion y la duracion real de la voz -- nadie escribe TSX a mano.
- Las llamadas al modelo **si son reales** -- cada agente invoca a Claude
  a traves del Agent SDK, usando la sesion de suscripcion de Claude Code
  (nunca una API key; ver `agents/_sdk_helpers.py`).
- QA y Render son deliberadamente **codigo Python puro**, no un LLM (ver
  `agents/qa_agent.py` y `agents/render_agent.py` para la justificacion).


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from state import ReelState
from agents._sdk_helpers import ejecutar

print("Repo root:", ROOT)


## 1. Perfilamiento y memoria compartida

Cada agente tiene un rol fijo, escrito en `prompts/*.txt` (perfilamiento,
ver ARCHITECTURE.md). La memoria del sistema es un solo objeto, `ReelState`,
que cada agente lee y actualiza -- no hay memoria entre corridas del
notebook, solo dentro de esta ejecucion.

El `brief` de abajo es el unico dato que inventamos nosotros (el tema y la
duracion deseada); todo lo demas lo produce el sistema de agentes.

In [ ]:
brief = {
    "tema": "ahorro_voluntario",
    "cliente_demo": "Cooperativa Demo (dato ficticio para esta demo)",
    "duracion_objetivo_s": 20,
}
state = ReelState(brief=brief)
state


## 2. Planificacion -- el Orquestador decide el plan de fases

In [ ]:
from agents import orchestrator

plan_resultado = ejecutar(orchestrator.planificar(state))
plan_resultado


## 3. Accion -- Investigacion

Este agente solo puede llamar a una herramienta: `leer_fuente`. Es una
fuente de datos simulada (en el motor real seria leer codigo o documentos
del cliente) -- si el tema no esta ahi, el agente debe decirlo, no
inventar.

In [ ]:
from agents import research_agent

inv_resultado = ejecutar(research_agent.investigar(state, tema=brief["tema"]))
inv_resultado


## 4. Guion -- convierte el dossier verificado en una tabla de escenas

In [ ]:
from agents import scriptwriter_agent

guion_resultado = ejecutar(scriptwriter_agent.escribir_guion(
    state, duracion_objetivo_s=brief["duracion_objetivo_s"]
))
guion_resultado


## 5. Visual (mock) y Voz (REAL)

El agente Visual decide kit/idea por escena con una herramienta mockeada
(no genera TSX -- la composicion `AgenteReel` ya es generica). El agente
de Voz llama UNA vez a `generar_voz_real`, que invoca OmniVoice de verdad
con todas las escenas del guion.

**Precondicion** (ver CLAUDE.md del motor de reels): esta maquina tiene
poca memoria virtual. Antes de correr la celda de voz, cierra WSL y
Chrome -- si no, la celda puede fallar o colgarse. La corrida real tarda
varios minutos (carga del modelo + una linea por escena).

In [ ]:
from agents import visual_agent

visual_resultado = ejecutar(visual_agent.generar_visual(state, kit="cartoon"))
visual_resultado


In [ ]:
from agents import voice_agent

voz_resultado = ejecutar(voice_agent.generar_voz(state))
voz_resultado


## 6. QA -- chequeo deterministico (no es una llamada a un LLM)

Compara la duracion total estimada (sumando lo que devolvio la herramienta
de voz, no una suposicion del LLM) contra la duracion objetivo, y verifica
que las escenas del guion y de la voz coincidan.

In [ ]:
from agents import qa_agent

reporte_qa = qa_agent.verificar(state, duracion_objetivo_s=brief["duracion_objetivo_s"])
reporte_qa


## 7. Replanificacion condicional

Si el QA no aprueba, el Orquestador decide como corregir y se repite el
ciclo Guion -> Voz -> QA una sola vez (limite para que la demo termine en
un numero acotado de llamadas).

In [ ]:
llamadas = [plan_resultado, inv_resultado, guion_resultado, voz_resultado, visual_resultado]

if not reporte_qa["aprobado"] and state.intentos_replanificacion < 1:
    replan_resultado = ejecutar(orchestrator.replanificar(
        state, observaciones="; ".join(reporte_qa["problemas"])
    ))
    llamadas.append(replan_resultado)
    instruccion = replan_resultado["decision"].get("instruccion", "")

    guion_resultado = ejecutar(scriptwriter_agent.escribir_guion(
        state, duracion_objetivo_s=brief["duracion_objetivo_s"], observaciones=instruccion
    ))
    llamadas.append(guion_resultado)

    voz_resultado = ejecutar(voice_agent.generar_voz(state))
    llamadas.append(voz_resultado)

    reporte_qa = qa_agent.verificar(state, duracion_objetivo_s=brief["duracion_objetivo_s"])
    print("QA tras replanificacion:", reporte_qa)
else:
    msg = "QA aprobado en el primer intento." if reporte_qa["aprobado"] else "Se agoto el intento de replanificacion."
    print(msg)


## 8. Render -- produce el MP4 real (codigo puro, no un LLM)

Ensambla los `props` de `AgenteReel` con el guion final y la duracion REAL
de cada escena (lo que acaba de medir OmniVoice, no el estimado original),
y corre `npx remotion render` en el repo publico `3d-animations-showcase`.
Puede tardar uno o dos minutos.

In [ ]:
from agents import render_agent

reporte_render = render_agent.renderizar(state)
{k: v for k, v in reporte_render.items() if k not in ("stdout_tail", "stderr_tail")}


In [ ]:
if not reporte_render["exitoso"]:
    print("STDERR:\n", reporte_render["stderr_tail"])
else:
    from IPython.display import Video, display

    display(Video(reporte_render["video"], embed=True, width=300))


## 9. Memoria final (`ReelState`) y bitacora de acciones

In [ ]:
print(state.resumen_json())


## 10. Costo real medido

No es un estimado: es la suma de lo que el SDK reporto en cada llamada
(ver `ResultMessage.usage` / `total_cost_usd` en `agents/_sdk_helpers.py`).

In [ ]:
total_input = sum((l.get("uso") or {}).get("input_tokens", 0) for l in llamadas)
total_output = sum((l.get("uso") or {}).get("output_tokens", 0) for l in llamadas)
total_costo = sum(l.get("costo_usd", 0) or 0 for l in llamadas)

print(f"Llamadas al modelo: {len(llamadas)}")
print(f"Tokens de entrada: {total_input}")
print(f"Tokens de salida: {total_output}")
print(f"Costo medido por el SDK: ${total_costo:.4f} USD")
